# 04 — Screenshot Rendering

Render HTML pages (from WARC extraction) to screenshots using Playwright.

**Important**: We render *saved HTML*, not live pages. No proxy needed, no live scraping.
External resources (CDN CSS/JS/images) will fail to load — this is expected. The model
learns to work with imperfect renders, which mirrors real-world conditions.

**Setup**: Install playwright browsers first:  
```bash
pip install playwright && playwright install chromium
```

**Output**: `data/screenshots/` — one PNG per URL (1280x800)

In [ ]:
import sys
sys.path.insert(0, '..')

import asyncio
import json
from pathlib import Path

SCREENSHOT_DIR = Path('../data/screenshots')
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)

# Verify Playwright is installed
try:
    from playwright.async_api import async_playwright
    print('Playwright installed')
except ImportError:
    print('Install: pip install playwright && playwright install chromium')

## Test Single Render

In [ ]:
from src.screenshot import render_screenshot
from IPython.display import Image as IPImage

# Test with a simple HTML page
test_html = """
<!DOCTYPE html>
<html>
<head><title>Murphy's Restaurant</title></head>
<body style="font-family: sans-serif; padding: 20px;">
  <h1>Murphy's Restaurant</h1>
  <p>Award-winning Irish cuisine in the heart of Dublin</p>
  <p><strong>Address:</strong> 42 Grafton Street, Dublin 2, Ireland</p>
  <p><strong>Phone:</strong> +353 1 234 5678</p>
  <p><strong>Hours:</strong> Mon-Sat 12pm-10pm, Sun 1pm-9pm</p>
  <p><strong>Cuisine:</strong> Irish, Contemporary</p>
  <p>⭐⭐⭐⭐⭐ 4.8/5 from 312 reviews</p>
</body>
</html>
"""

test_out = SCREENSHOT_DIR / 'test_render.png'
success = await render_screenshot(test_html, str(test_out))
print(f'Render success: {success}')

if success:
    IPImage(str(test_out))

## Batch Render from WARC Manifest

In [ ]:
# Load manifest from notebook 03
manifest_path = Path('../data/raw/warc_manifest.jsonl')

if not manifest_path.exists():
    print('Run notebook 03 first to generate warc_manifest.jsonl')
else:
    manifest = []
    with open(manifest_path) as f:
        for line in f:
            manifest.append(json.loads(line))
    print(f'Loaded {len(manifest)} manifest entries')

In [ ]:
from src.screenshot import batch_render

# Build items list for batch render
render_items = []
for entry in manifest:
    html_path = entry.get('html_path')
    if html_path and Path(html_path).exists():
        with open(html_path, 'r', encoding='utf-8', errors='replace') as f:
            html = f.read()
        
        # Use domain as ID for the screenshot filename
        from urllib.parse import urlparse
        domain = urlparse(entry['url']).netloc.lstrip('www.')
        
        render_items.append({'id': domain, 'html': html})

print(f'Pages to render: {len(render_items)}')

In [ ]:
# Batch render with 8 concurrent workers
# Expected throughput: ~2,000-4,000 screenshots/hour

if render_items:
    results = await batch_render(
        items=render_items,
        output_dir=str(SCREENSHOT_DIR),
        concurrency=8,
        skip_existing=True,
    )
    success_count = sum(v for v in results.values())
    print(f'\nRendered: {success_count}/{len(render_items)} pages successfully')
    print(f'Failed: {len(render_items) - success_count}')
else:
    print('No items to render — run notebooks 02 and 03 first')

## Quality Check: Screenshot Inspection

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import random

# Display a random sample of screenshots
screenshots = list(SCREENSHOT_DIR.glob('*.png'))
screenshots = [s for s in screenshots if s.name != 'test_render.png']

print(f'Total screenshots: {len(screenshots)}')

if screenshots:
    sample = random.sample(screenshots, min(6, len(screenshots)))
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    
    for ax, path in zip(axes, sample):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.set_title(path.stem[:40], fontsize=8)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Update manifest with screenshot paths
updated_manifest = []
for entry in manifest:
    from urllib.parse import urlparse
    domain = urlparse(entry['url']).netloc.lstrip('www.')
    screenshot_path = SCREENSHOT_DIR / f'{domain}.png'
    entry['screenshot_path'] = str(screenshot_path) if screenshot_path.exists() else None
    updated_manifest.append(entry)

with open(manifest_path, 'w') as f:
    for entry in updated_manifest:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

with_screenshots = sum(1 for e in updated_manifest if e.get('screenshot_path'))
print(f'Manifest updated: {with_screenshots}/{len(updated_manifest)} entries have screenshots')
print('\nNext step: 05_training_data_prep.ipynb')